## Phase 1, Step 1: Ingestion & Optimization

In [0]:
import zipfile
import pandas as pd
import numpy as np

# 1. Path to your uploaded volume
volume_path = "/Volumes/workspace/default/m5_walmart_data"
# Change this to match the EXACT name of the file you see in your Volume
zip_file_path = "/Volumes/workspace/default/m5_walmart_data/m5-forecasting-accuracy.zip"
# 2. Memory Optimization Function (as per your plan)
def reduce_mem_usage(df):
    for col in df.columns:
        if df[col].dtype != object:
            c_min, c_max = df[col].min(), df[col].max()
            if str(df[col].dtype)[:3] == 'int':
                if c_min > np.iinfo(np.int8).min and c_max < np.iinfo(np.int8).max:
                    df[col] = df[col].astype(np.int8)
                else: df[col] = df[col].astype(np.int16)
            else:
                df[col] = df[col].astype(np.float32) # float64 to float32
        else:
            df[col] = df[col].astype('category')
    return df

# 3. Extracting and Loading
with zipfile.ZipFile(zip_file_path, 'r') as z:
    # Load the main sales file
    with z.open('sales_train_evaluation.csv') as f:
        sales = pd.read_csv(f)
        sales = reduce_mem_usage(sales)

print("M5 Data Loaded and Optimized.")

## Phase 1, Step 1.2: Reshaping to Long Format (Melt)

In [0]:
import gc

# Reshape from 1,913 day-columns to a single 'sales' column
sales_long = pd.melt(sales, 
                     id_vars=['id', 'item_id', 'dept_id', 'cat_id', 'store_id', 'state_id'], 
                     var_name='day', 
                     value_name='sales')

# Immediate memory management: Clear the old wide dataframe to free up RAM
del sales
gc.collect()

print("Data reshaped to Long Format. Ready for Signal Engineering.")

## Phase 1, Step 2: Signal Engineering

In [0]:
import gc
import numpy as np

# 1. Feature List from your Project Plan
lags = [7, 28] # We will do 365 separately if this passes
windows = [7, 28]

# 2. Iterative Signal Engineering to save RAM
for lag in lags:
    print(f"Processing Lag {lag}...")
    sales_long[f'lag_{lag}'] = sales_long.groupby('id')['sales'].transform(lambda x: x.shift(lag)).astype(np.float32)
    gc.collect()

for window in windows:
    print(f"Processing Rolling Mean {window}...")
    # Shift(1) is critical to avoid data leakage as per plan
    sales_long[f'rolling_mean_{window}'] = sales_long.groupby('id')['sales'].transform(
        lambda x: x.shift(1).rolling(window).mean()
    ).astype(np.float32)
    gc.collect()

# 3. Drop rows where we don't have enough history (the 'NaN' buffer)
sales_long = sales_long.dropna(subset=['lag_28', 'rolling_mean_28'])
gc.collect()

print("Signal Engineering Complete without crashing the kernel.")

## Phase 1, Step 2.2: Contextual Joins (Calendar & Prices)

In [0]:
import pandas as pd
import zipfile
import gc
import numpy as np

# 1. Join Calendar Hooks (SNAP, Holidays)
cal_cols = ['d', 'wm_yr_wk', 'wday', 'month', 'year', 'snap_CA', 'snap_TX', 'snap_WI']
zip_path = "/Volumes/workspace/default/m5_walmart_data/m5-forecasting-accuracy.zip"

with zipfile.ZipFile(zip_path, 'r') as z:
    with z.open('calendar.csv') as f: 
        calendar = pd.read_csv(f, usecols=cal_cols)
        calendar = reduce_mem_usage(calendar) # Keep using the memory optimization!

sales_long = sales_long.merge(calendar, left_on='day', right_on='d', how='left')
sales_long = sales_long.drop(columns=['d'])
del calendar
gc.collect()

# 2. Join Price Signal
with zipfile.ZipFile(zip_path, 'r') as z:
    with z.open('sell_prices.csv') as f:
        prices = pd.read_csv(f)
        prices = reduce_mem_usage(prices)

sales_long = sales_long.merge(prices, on=['store_id', 'item_id', 'wm_yr_wk'], how='left')
del prices
gc.collect()

print("M5 Silver Layer (Sales + Calendar + Prices) ready for the Model.")

## Phase 1, Step 3: Data Persistence (Silver Layer)

In [0]:
# Save the Silver Layer as a permanent Parquet file
file_path = '/Volumes/workspace/default/m5_walmart_data/m5_silver_layer.parquet'
sales_long.to_parquet(file_path, index=False)

print(f"Silver Layer successfully persisted to: {file_path}")